# 05. 무패턴 피처 — 팀원 신규 8종(12열) + 같은 분 같은 쌍 6열

06 LightGBM 에서 기존 v10 피처(`features_{train,val,test}`)에 **붙일 열**을 만든다. 출력은 분할 전체 거래, `edge_id` 기준.

| 묶음 | 열 | 근거 |
|---|---|---|
| 팀원 신규 8종 · 12열 | V10 이진모델 개선 기록(HTML) 정의대로 구현. 정의가 애매한 곳은 아래 **정한 정의** 표에 적었다 | 팀원 41종 채택본 |
| 같은 분 같은 쌍 · 6열 | 쌍둥이 칸(새 쌍이 같은 분에 2건 → 양성 1 + 정상 1, 양성 쪽 금액이 큰 경우 70%) | 04 진단 |

- 계산에는 **라벨을 쓰지 않는다**. 라벨은 마지막 점검 셀에서만 붙인다
- 이력: 2022-08-01 부터. 대상: Train 8/8~10/2 · Val 10/3~10/16 · Test 10/17~10/30
- 시점: "전날까지" 피처는 대상일 0시 이전 거래만, "당일" 피처는 대상일 마감까지(당일 나중 거래 포함), "같은 분" 피처는 그 분 전체. 팀원 운영 조건(당일 마감 판정)과 같다

## 피처 정의

### 팀원 신규 8종 (12열)

| 열 | HTML 정의 | 정한 정의 (애매한 부분) |
|---|---|---|
| `minute_sender_vs_prior_hour` | log1p(60 × 동일 분 비자기 송금 건수) − log1p(이전 60분 송금 건수) | "비자기" = **현재 거래 자신 제외**(같은 분 송신 계좌의 다른 송금 수). 자기이체도 센다. 이전 60분 = `[ts−60분, ts−1분]` |
| `minute_receiver_vs_prior_hour` | log1p(60 × 동일 분 비자기 수취 건수) − log1p(이전 60분 수취 건수) | 위와 같은 규칙, 수신 계좌 기준 |
| `src_day_ccy_flow_balance` | 계좌·통화별 당일 수취액과 송금액의 min/max, 양쪽 0이면 0 | 송신 계좌, **지급통화** 기준: 당일 송금 지급액 합 vs 당일 입금 수취액 합(입금은 수취통화가 그 통화인 거래). 자기이체는 양쪽에 다 들어감 |
| `dst_day_ccy_flow_balance` | 〃 | 수신 계좌, **수취통화** 기준 |
| `src_day_ccy_log_matching_in_amount` | 송신 계좌·지급통화·지급액과 같은 금액이 당일 들어온 횟수의 log1p | 당일 송신 계좌가 받은 거래 중 수취통화 = 이 거래 지급통화, 수취액 = 이 거래 지급액인 건수. 시각 순서 무관(당일 전체), 자기이체면 자신도 포함 |
| `src_prior_ccy_amount_seen` | 계좌·통화·방향별 대상일 이전 동일 금액 관측 여부 | 송신 계좌가 (지급통화, 지급액)을 **전날까지** 보낸 적 있으면 1 |
| `dst_prior_ccy_amount_seen` | 〃 | 수신 계좌가 (수취통화, 수취액)을 전날까지 받은 적 있으면 1 |
| `src_prior_ccy_amount_vs_mean` | log1p(대상 금액 / 같은 계좌·통화·방향의 이전 날짜 평균 금액), 없으면 0 | 지급액 / 송신 계좌의 그 지급통화 송금 평균(전날까지) |
| `dst_prior_ccy_amount_vs_mean` | 〃 | 수취액 / 수신 계좌의 그 수취통화 입금 평균(전날까지) |
| `src_day_ccy_rate_vs_prior` | log1p(당일 방향·통화 건수) − log1p(이전 건수 / 최초 과거 활동일부터의 달력 일수) | 달력 일수 = 대상일 − (그 계좌·통화·방향의 첫 활동일). 이전 건수 0이면 뒤 항 0 |
| `dst_day_ccy_rate_vs_prior` | 〃 | 수신 방향·수취통화 |
| `pair_amount_seen_before_day` | 같은 src, dst, 지급통화, 지급액이 대상일 이전에 관측됐는지 | 그대로 (전날까지) |

### 같은 분 같은 쌍 (6열, 04 진단에서)

| 열 | 정의 |
|---|---|
| `pair_same_minute_other_n` | 같은 분 같은 쌍(src, dst) 다른 거래 수 |
| `pair_same_minute_log_amount_ratio` | ln(1+지급액) − 같은 분 같은 쌍 **같은 지급통화** 다른 거래들의 ln(1+지급액) 평균. 다른 거래 없으면 0 |
| `pair_same_minute_is_max` | 같은 분 같은 쌍 같은 지급통화 거래가 2건 이상이고, 그중 지급액이 **단독** 최대면 1 |
| `pair_count_7d` | ln(1 + 같은 쌍의 `[ts−7일, ts−1분]` 거래 수). 누적 `pair_prior_count` 는 이력 시작(8/1)에 따라 train 초반에 작게 잡혀 기간별로 흔들리므로 고정 창 버전을 둔다 |
| `receiver_same_minute_other_in_n` | 같은 분 수신 계좌의 다른 입금 수 (03 정의와 같음) |
| `receiver_same_minute_other_in_neighbors` | 같은 분 수신 계좌에 보낸 다른 송신 계좌 수 (03 정의와 같음) |

금액 비교(같은 금액, 평균 대비)는 **같은 통화 안에서만** 한다.

## 0. 설정

GPU 서버 Jupyter, `code/nonpaten_vs_normal` 에서 실행. 입력은 `code/01_prepare_parquet.ipynb` 출력(`transactions.parquet`; 점검 셀만 `patterns.parquet`).

출력:

```text
parquets/HI-Large/nonpaten_vs_normal/05_features/
├─ train/2022-08-08.parquet … (날짜별, edge_id + 18열)
├─ val/…, test/…
├─ features.json        정의·건수·점검 결과
└─ work.duckdb          중간 테이블 (다시 실행 시 재사용, 다 끝나면 지워도 됨)
```

날짜별 파일이 이미 있으면 건너뛴다. 커널이 죽으면 그대로 다시 실행하면 된다.

In [1]:
# 현재 Jupyter 커널에 패키지가 없을 때만 주석을 해제해 실행합니다.
# !pip install --break-system-packages --no-cache-dir -q duckdb pandas

In [2]:
from pathlib import Path
import json, time
from datetime import date, timedelta
import duckdb, numpy as np, pandas as pd
from IPython.display import display

DATASET = "HI-Large"
PROJECT_ROOT = None            # 자동 탐색 실패 시 Path("/workspace/money_laundry_ai_")
THREADS, MEMORY = 4, "8GB"
REBUILD = False                # True: 중간 테이블·날짜별 파일 새로 만들기

HISTORY = ("2022-08-01", "2022-10-31")   # 이력 [시작, 끝)
SPLITS = {"train": ("2022-08-08", "2022-10-02"), "val": ("2022-10-03", "2022-10-16"), "test": ("2022-10-17", "2022-10-30")}
EXPECTED_ROWS = {"train": 102_336_629, "val": 24_413_046, "test": 26_808_230}   # binary_classification 실험과 같은 거래 수
STRICT = True                  # 구간별 거래 수가 다르면 멈춤

TEAM_FEATURES = [
    "minute_sender_vs_prior_hour", "minute_receiver_vs_prior_hour",
    "src_day_ccy_flow_balance", "dst_day_ccy_flow_balance",
    "src_day_ccy_log_matching_in_amount",
    "src_prior_ccy_amount_seen", "dst_prior_ccy_amount_seen",
    "src_prior_ccy_amount_vs_mean", "dst_prior_ccy_amount_vs_mean",
    "src_day_ccy_rate_vs_prior", "dst_day_ccy_rate_vs_prior",
    "pair_amount_seen_before_day",
]
PAIR_FEATURES = [
    "pair_same_minute_other_n", "pair_same_minute_log_amount_ratio", "pair_same_minute_is_max",
    "pair_count_7d", "receiver_same_minute_other_in_n", "receiver_same_minute_other_in_neighbors",
]
FEATURES = TEAM_FEATURES + PAIR_FEATURES
assert len(TEAM_FEATURES) == 12 and len(PAIR_FEATURES) == 6


def find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "parquets" / DATASET / "transactions.parquet").is_file():
            return c
    raise FileNotFoundError("transactions.parquet 를 찾지 못했습니다. PROJECT_ROOT 를 지정하세요.")


ROOT = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else find_root()
BASE = ROOT / "parquets" / DATASET
TX, PAT = BASE / "transactions.parquet", BASE / "patterns.parquet"
OUT = BASE / "nonpaten_vs_normal" / "05_features"
assert TX.is_file(), TX
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("OUT :", OUT)

ROOT: /workspace/money_laundry_ai_
OUT : /workspace/money_laundry_ai_/parquets/HI-Large/nonpaten_vs_normal/05_features


In [3]:
con = duckdb.connect(str(OUT / "work.duckdb"))
con.execute(f"SET threads={THREADS}")
con.execute(f"SET memory_limit='{MEMORY}'")
con.execute(f"SET temp_directory='{OUT / 'tmp'}'")
con.execute("SET preserve_insertion_order=false")


def q(v):
    return "'" + str(v).replace("'", "''") + "'"


def has(name):
    return con.execute("SELECT count(*) FROM duckdb_tables() WHERE table_name = ?", [name]).fetchone()[0] > 0


def step(name, sql):
    """CREATE TABLE name AS sql — 이미 있으면 건너뜀"""
    if has(name) and not REBUILD:
        print(f"{name:12s} 이미 있음 — 건너뜀")
        return
    t = time.perf_counter()
    con.execute(f"CREATE OR REPLACE TABLE {name} AS {sql}")
    n = con.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
    print(f"{name:12s} {n:>13,} 행  ({time.perf_counter() - t:,.1f}s)", flush=True)


def days(a, b):
    d0, d1 = date.fromisoformat(a), date.fromisoformat(b)
    return [str(d0 + timedelta(i)) for i in range((d1 - d0).days + 1)]

## 1. 거래 `tx` + 금액 첫 관측일

라벨 열은 읽지 않는다. 같은 금액을 "전날까지 본 적 있나"는 **그 키의 첫 관측일 < 대상일** 과 같으므로 윈도 `min(day)` 로 붙인다.

- `src_amt_first`: (src, 지급통화, 지급액) 첫 날 · `dst_amt_first`: (dst, 수취통화, 수취액) 첫 날 · `pair_amt_first`: (src, dst, 지급통화, 지급액) 첫 날

In [4]:
step("tx", f"""
    SELECT edge_id, ts, day, src, dst, paid, received, payment_currency AS pc, receiving_currency AS rc,
           min(day) OVER (PARTITION BY src, payment_currency, paid)        AS src_amt_first,
           min(day) OVER (PARTITION BY dst, receiving_currency, received)  AS dst_amt_first,
           min(day) OVER (PARTITION BY src, dst, payment_currency, paid)   AS pair_amt_first
    FROM read_parquet({q(TX)})
    WHERE ts >= TIMESTAMP {q(HISTORY[0])} AND ts < TIMESTAMP {q(HISTORY[1])}
""")
con.execute("SELECT * FROM tx LIMIT 5").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

tx             168,490,912 행  (347.8s)


,edge_id,ts,day,src,dst,paid,received,pc,rc,src_amt_first,dst_amt_first,pair_amt_first
0,108257833,2022-09-28 23:31:00,2022-09-28,513662,493726,53808.81,53808.81,Yuan,Yuan,2022-08-01,2022-08-01,2022-08-01
1,20104489,2022-08-10 16:07:00,2022-08-10,513662,493726,53808.81,53808.81,Yuan,Yuan,2022-08-01,2022-08-01,2022-08-01
2,40549829,2022-08-22 14:58:00,2022-08-22,513662,493726,53808.81,53808.81,Yuan,Yuan,2022-08-01,2022-08-01,2022-08-01
3,37176296,2022-08-19 20:01:00,2022-08-19,513662,493726,53808.81,53808.81,Yuan,Yuan,2022-08-01,2022-08-01,2022-08-01
4,85402255,2022-09-15 18:37:00,2022-09-15,513662,493726,53808.81,53808.81,Yuan,Yuan,2022-08-01,2022-08-01,2022-08-01


## 2. 같은 분 집계

- `mp` (src, dst, 분): 건수 `n`, 쌍의 직전 7일 건수 `prior7` (`[ts−7일, ts−1분]`)
- `mpc` (src, dst, 분, 지급통화) — 2건 이상인 칸만: 건수, ln(1+지급액) 합, 최대 지급액, 최대 금액 건수
- `mo` (src, 분) 송신 건수 + 직전 60분 · `mi` (dst, 분) 수신 건수·송신 계좌 수 + 직전 60분

In [5]:
step("mp", """
    SELECT src, dst, ts, n,
           coalesce(sum(n) OVER (PARTITION BY src, dst ORDER BY ts
                                 RANGE BETWEEN INTERVAL 7 DAY PRECEDING AND INTERVAL 1 MINUTE PRECEDING), 0) AS prior7
    FROM (SELECT src, dst, ts, count(*) AS n FROM tx GROUP BY ALL)
""")
step("mpc", """
    WITH a AS (SELECT t.src, t.dst, t.ts, t.pc, t.paid, count(*) AS c
               FROM tx t JOIN (SELECT src, dst, ts FROM mp WHERE n > 1) m
                 ON m.src = t.src AND m.dst = t.dst AND m.ts = t.ts
               GROUP BY ALL)
    SELECT src, dst, ts, pc, sum(c) AS n, sum(c * ln(1 + paid)) AS sum_l, max(paid) AS max_paid, arg_max(c, paid) AS cnt_at_max
    FROM a GROUP BY ALL HAVING sum(c) > 1
""")
step("mo", """
    SELECT src, ts, n,
           coalesce(sum(n) OVER (PARTITION BY src ORDER BY ts
                                 RANGE BETWEEN INTERVAL 60 MINUTE PRECEDING AND INTERVAL 1 MINUTE PRECEDING), 0) AS ph
    FROM (SELECT src, ts, sum(n) AS n FROM mp GROUP BY ALL)
""")
step("mi", """
    SELECT dst, ts, n, neighbors,
           coalesce(sum(n) OVER (PARTITION BY dst ORDER BY ts
                                 RANGE BETWEEN INTERVAL 60 MINUTE PRECEDING AND INTERVAL 1 MINUTE PRECEDING), 0) AS ph
    FROM (SELECT dst, ts, sum(n) AS n, count(*) AS neighbors FROM mp GROUP BY ALL)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mp             164,929,785 행  (56.6s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mpc              3,522,045 행  (6.4s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mo             148,301,973 행  (48.6s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mi             163,975,984 행  (53.8s)


## 3. 당일 계좌·통화 집계

- `out_day` (계좌=src, 날짜, 지급통화): 당일 건수·지급액 합, **전날까지** 누적 건수·금액, 첫 활동일
- `in_day` (계좌=dst, 날짜, 수취통화): 같은 값, 수취액 기준
- `in_amt` (계좌=dst, 날짜, 수취통화, 수취액): 건수 — 당일 같은 금액 입금 횟수용

In [6]:
for name, acct, ccy, amt in [("out_day", "src", "pc", "paid"), ("in_day", "dst", "rc", "received")]:
    step(name, f"""
        SELECT acct, day, ccy, cnt, amt,
               coalesce(sum(cnt) OVER w, 0) AS prior_cnt, coalesce(sum(amt) OVER w, 0) AS prior_amt,
               min(day) OVER (PARTITION BY acct, ccy) AS first_day
        FROM (SELECT {acct} AS acct, day, {ccy} AS ccy, count(*) AS cnt, sum({amt}) AS amt FROM tx GROUP BY ALL)
        WINDOW w AS (PARTITION BY acct, ccy ORDER BY day ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
    """)
step("in_amt", "SELECT dst AS acct, day, rc AS ccy, received AS amt, count(*) AS c FROM tx GROUP BY ALL")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

out_day         49,746,143 행  (36.4s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

in_day          64,480,720 행  (44.4s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

in_amt         168,476,521 행  (33.4s)


## 4. 피처 조립 — 날짜별 parquet

정의 표의 식을 그대로 옮긴 SQL. 날짜 하나씩 `tx` 를 잘라 붙인다.

In [7]:
def feature_sql(day):
    D = f"DATE {q(day)}"
    T = f"ts >= TIMESTAMP {q(day)} AND ts < {D} + INTERVAL 1 DAY"
    bal = lambda o, i: f"CASE WHEN greatest({o}, {i}) > 0 THEN least({o}, {i}) / greatest({o}, {i}) ELSE 0 END"
    rate = lambda a: (f"ln(1 + {a}.cnt) - CASE WHEN {a}.prior_cnt > 0 "
                      f"THEN ln(1 + {a}.prior_cnt / date_diff('day', {a}.first_day, t.day)) ELSE 0 END")
    vs_mean = lambda a, x: (f"CASE WHEN {a}.prior_cnt > 0 AND {a}.prior_amt > 0 "
                            f"THEN ln(1 + {x} / ({a}.prior_amt / {a}.prior_cnt)) ELSE 0 END")
    expr = {
        "minute_sender_vs_prior_hour":   "ln(1 + 60 * (mo.n - 1)) - ln(1 + mo.ph)",
        "minute_receiver_vs_prior_hour": "ln(1 + 60 * (mi.n - 1)) - ln(1 + mi.ph)",
        "src_day_ccy_flow_balance": bal("os.amt", "coalesce(i_s.amt, 0)"),
        "dst_day_ccy_flow_balance": bal("coalesce(o_d.amt, 0)", "id.amt"),
        "src_day_ccy_log_matching_in_amount": "ln(1 + coalesce(m.c, 0))",
        "src_prior_ccy_amount_seen": "(t.src_amt_first < t.day)::INT",
        "dst_prior_ccy_amount_seen": "(t.dst_amt_first < t.day)::INT",
        "src_prior_ccy_amount_vs_mean": vs_mean("os", "t.paid"),
        "dst_prior_ccy_amount_vs_mean": vs_mean("id", "t.received"),
        "src_day_ccy_rate_vs_prior": rate("os"),
        "dst_day_ccy_rate_vs_prior": rate("id"),
        "pair_amount_seen_before_day": "(t.pair_amt_first < t.day)::INT",
        "pair_same_minute_other_n": "mp.n - 1",
        "pair_same_minute_log_amount_ratio": "CASE WHEN mc.n > 1 THEN ln(1 + t.paid) - (mc.sum_l - ln(1 + t.paid)) / (mc.n - 1) ELSE 0 END",
        "pair_same_minute_is_max": "coalesce(mc.n > 1 AND t.paid = mc.max_paid AND mc.cnt_at_max = 1, false)::INT",
        "pair_count_7d": "ln(1 + mp.prior7)",
        "receiver_same_minute_other_in_n": "mi.n - 1",
        "receiver_same_minute_other_in_neighbors": "mi.neighbors - CASE WHEN mp.n = 1 THEN 1 ELSE 0 END",
    }
    assert list(expr) == FEATURES
    cols = ",\n           ".join(f'({v})::FLOAT AS "{k}"' for k, v in expr.items())
    return f"""
    SELECT t.edge_id,
           {cols}
    FROM (SELECT * FROM tx WHERE day = {D}) t
    JOIN (SELECT * FROM mp WHERE {T}) mp ON mp.src = t.src AND mp.dst = t.dst AND mp.ts = t.ts
    JOIN (SELECT * FROM mo WHERE {T}) mo ON mo.src = t.src AND mo.ts = t.ts
    JOIN (SELECT * FROM mi WHERE {T}) mi ON mi.dst = t.dst AND mi.ts = t.ts
    LEFT JOIN (SELECT * FROM mpc WHERE {T}) mc ON mc.src = t.src AND mc.dst = t.dst AND mc.ts = t.ts AND mc.pc = t.pc
    JOIN (SELECT * FROM out_day WHERE day = {D}) os ON os.acct = t.src AND os.ccy = t.pc
    LEFT JOIN (SELECT * FROM in_day WHERE day = {D}) i_s ON i_s.acct = t.src AND i_s.ccy = t.pc
    JOIN (SELECT * FROM in_day WHERE day = {D}) id ON id.acct = t.dst AND id.ccy = t.rc
    LEFT JOIN (SELECT * FROM out_day WHERE day = {D}) o_d ON o_d.acct = t.dst AND o_d.ccy = t.rc
    LEFT JOIN (SELECT * FROM in_amt WHERE day = {D}) m ON m.acct = t.src AND m.ccy = t.pc AND m.amt = t.paid
    """


print(feature_sql("2022-10-03")[:1500])


    SELECT t.edge_id,
           (ln(1 + 60 * (mo.n - 1)) - ln(1 + mo.ph))::FLOAT AS "minute_sender_vs_prior_hour",
           (ln(1 + 60 * (mi.n - 1)) - ln(1 + mi.ph))::FLOAT AS "minute_receiver_vs_prior_hour",
           (CASE WHEN greatest(os.amt, coalesce(i_s.amt, 0)) > 0 THEN least(os.amt, coalesce(i_s.amt, 0)) / greatest(os.amt, coalesce(i_s.amt, 0)) ELSE 0 END)::FLOAT AS "src_day_ccy_flow_balance",
           (CASE WHEN greatest(coalesce(o_d.amt, 0), id.amt) > 0 THEN least(coalesce(o_d.amt, 0), id.amt) / greatest(coalesce(o_d.amt, 0), id.amt) ELSE 0 END)::FLOAT AS "dst_day_ccy_flow_balance",
           (ln(1 + coalesce(m.c, 0)))::FLOAT AS "src_day_ccy_log_matching_in_amount",
           ((t.src_amt_first < t.day)::INT)::FLOAT AS "src_prior_ccy_amount_seen",
           ((t.dst_amt_first < t.day)::INT)::FLOAT AS "dst_prior_ccy_amount_seen",
           (CASE WHEN os.prior_cnt > 0 AND os.prior_amt > 0 THEN ln(1 + t.paid / (os.prior_amt / os.prior_cnt)) ELSE 0 END)::FLOAT AS "src_pr

In [8]:
rows = {}
for split, (a, b) in SPLITS.items():
    (OUT / split).mkdir(exist_ok=True)
    t0, n_split = time.perf_counter(), 0
    for i, day in enumerate(days(a, b)):
        f = OUT / split / f"{day}.parquet"
        if f.exists() and not REBUILD:
            n_split += con.execute(f"SELECT count(*) FROM read_parquet({q(f)})").fetchone()[0]
            continue
        part = f.with_suffix(".partial")
        con.execute(f"COPY ({feature_sql(day)}) TO {q(part)} (FORMAT PARQUET, COMPRESSION ZSTD)")
        n_day = con.execute(f"SELECT count(*) FROM read_parquet({q(part)})").fetchone()[0]
        n_tx = con.execute(f"SELECT count(*) FROM tx WHERE day = DATE {q(day)}").fetchone()[0]
        assert n_day == n_tx, f"{day}: 피처 {n_day:,} ≠ 거래 {n_tx:,} (조인 누락)"
        part.rename(f)
        n_split += n_day
        if i % 7 == 6:
            print(f"  {split} {day}  누적 {n_split:,}  ({time.perf_counter() - t0:,.0f}s)", flush=True)
    rows[split] = n_split
    print(f"{split}: {n_split:,} 행 (기대 {EXPECTED_ROWS[split]:,}) {'OK' if n_split == EXPECTED_ROWS[split] else '불일치'}"
          f"  {time.perf_counter() - t0:,.0f}s", flush=True)
assert rows == EXPECTED_ROWS or not STRICT, rows

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (86 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  test 2022-10-30  누적 26,808,230  (1,807s)
test: 26,808,230 행 (기대 26,808,230) OK  1,807s


## 5. 점검

1. 비유한값·값 범위
2. (라벨 사용) 분할별 정상 / 패턴 / 무패턴 평균
3. 팀원 HTML 수치와 비교 — 같은 정의인지 확인용

In [9]:
con.execute(f"""CREATE OR REPLACE VIEW feat AS
                SELECT *, '{list(SPLITS)[0]}' AS split FROM read_parquet({q(OUT / 'train' / '*.parquet')})
                UNION ALL SELECT *, 'val'  FROM read_parquet({q(OUT / 'val' / '*.parquet')})
                UNION ALL SELECT *, 'test' FROM read_parquet({q(OUT / 'test' / '*.parquet')})""")
bad = con.execute("SELECT " + ", ".join(f'count(*) FILTER (WHERE "{c}" IS NULL OR NOT isfinite("{c}")) AS "{c}"' for c in FEATURES)
                  + " FROM feat").df().T[0]
assert bad.sum() == 0, bad[bad > 0]
desc = con.execute("SELECT " + ", ".join(f'min("{c}"), avg("{c}"), max("{c}"), avg(("{c}" = 0)::INT)' for c in FEATURES) + " FROM feat").fetchone()
desc = pd.DataFrame(np.array(desc).reshape(-1, 4), index=FEATURES, columns=["min", "mean", "max", "zero_rate"]).astype(float).round(4)
desc

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min,mean,max,zero_rate
minute_sender_vs_prior_hour,-8.2443,-0.1250,6.0426,0.4160
minute_receiver_vs_prior_hour,-4.8752,-0.2237,6.2934,0.4742
src_day_ccy_flow_balance,0.0000,0.1654,1.0000,0.3375
dst_day_ccy_flow_balance,0.0000,0.1424,1.0000,0.5348
src_day_ccy_log_matching_in_amount,0.0000,0.0382,2.9444,0.9450
src_prior_ccy_amount_seen,0.0000,0.9221,1.0000,0.0779
dst_prior_ccy_amount_seen,0.0000,0.9220,1.0000,0.0780
src_prior_ccy_amount_vs_mean,0.0000,0.4107,20.9744,0.0012
dst_prior_ccy_amount_vs_mean,0.0000,0.4504,28.2731,0.0018
src_day_ccy_rate_vs_prior,-2.4794,0.1559,2.3753,0.0380


In [10]:
# 라벨은 여기서만 붙인다: cls 0 정상 · 1 패턴 · 2 무패턴
con.execute(f"""CREATE OR REPLACE TABLE labels AS
    SELECT t.edge_id, t.payment_format AS fmt, t.day,
           (CASE WHEN NOT t.is_laundering THEN 0 WHEN p.edge_id IS NOT NULL THEN 1 ELSE 2 END)::TINYINT AS cls
    FROM read_parquet({q(TX)}) t
    LEFT JOIN (SELECT DISTINCT edge_id FROM read_parquet({q(PAT)}) WHERE NOT unmatched) p ON p.edge_id = t.edge_id
    WHERE t.day BETWEEN DATE {q(SPLITS['train'][0])} AND DATE {q(SPLITS['test'][1])}""")
means = con.execute("SELECT split, cls, count(*) AS n, " + ", ".join(f'avg("{c}") AS "{c}"' for c in FEATURES)
                    + " FROM feat JOIN labels USING (edge_id) GROUP BY ALL ORDER BY split, cls").df()
means.set_index(["split", "cls"]).T.round(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

split                                            test                        \
cls                                                 0          1          2   
n                                        2.677377e+07  20187.000  14273.000   
minute_sender_vs_prior_hour             -1.200000e-01      2.447      0.310   
minute_receiver_vs_prior_hour           -2.130000e-01      0.345      0.140   
src_day_ccy_flow_balance                 1.940000e-01      0.641      0.091   
dst_day_ccy_flow_balance                 1.720000e-01      0.166      0.080   
src_day_ccy_log_matching_in_amount       6.100000e-02      0.353      0.005   
src_prior_ccy_amount_seen                9.060000e-01      0.000      0.251   
dst_prior_ccy_amount_seen                9.060000e-01      0.000      0.252   
src_prior_ccy_amount_vs_mean             4.260000e-01      0.516      0.396   
dst_prior_ccy_amount_vs_mean             4.770000e-01      1.337      0.950   
src_day_ccy_rate_vs_prior                1.840000e-01      0.571      0.469   
dst_day_ccy_rate_vs_prior                2.290000e-01      0.458      0.956   
pair_amount_seen_before_day              9.060000e-01      0.000      0.251   
pair_same_minute_other_n                 4.100000e-02      0.092      0.094   
pair_same_minute_log_amount_ratio       -0.000000e+00      0.160     -0.078   
pair_same_minute_is_max                  2.000000e-02      0.073      0.032   
pair_count_7d                            2.155000e+00      0.118      0.753   
receiver_same_minute_other_in_n          5.600000e-02      0.094      0.245   
receiver_same_minute_other_in_neighbors  5.600000e-02      0.094      0.245   

split                                           train                        \
cls                                                 0          1          2   
n                                        1.022260e+08  62889.000  47782.000   
minute_sender_vs_prior_hour             -1.290000e-01      2.444      0.143   
minute_receiver_vs_prior_hour           -2.270000e-01      0.327     -0.017   
src_day_ccy_flow_balance                 1.650000e-01      0.636      0.075   
dst_day_ccy_flow_balance                 1.430000e-01      0.151      0.074   
src_day_ccy_log_matching_in_amount       3.800000e-02      0.357      0.003   
src_prior_ccy_amount_seen                9.230000e-01      0.000      0.289   
dst_prior_ccy_amount_seen                9.230000e-01      0.000      0.289   
src_prior_ccy_amount_vs_mean             4.120000e-01      0.553      0.460   
dst_prior_ccy_amount_vs_mean             4.490000e-01      1.396      1.067   
src_day_ccy_rate_vs_prior                1.560000e-01      0.553      0.397   
dst_day_ccy_rate_vs_prior                2.090000e-01      0.420      0.838   
pair_amount_seen_before_day              9.230000e-01      0.000      0.289   
pair_same_minute_other_n                 4.300000e-02      0.088      0.055   
pair_same_minute_log_amount_ratio       -0.000000e+00      0.183     -0.047   
pair_same_minute_is_max                  2.100000e-02      0.075      0.019   
pair_count_7d                            2.250000e+00      0.131      0.829   
receiver_same_minute_other_in_n          6.000000e-02      0.089      0.175   
receiver_same_minute_other_in_neighbors  5.900000e-02      0.089      0.174   

split                                             val                        
cls                                                 0          1          2  
n                                        2.437925e+07  20100.000  13695.000  
minute_sender_vs_prior_hour             -1.270000e-01      2.484      0.254  
minute_receiver_vs_prior_hour           -2.260000e-01      0.351      0.087  
src_day_ccy_flow_balance                 1.360000e-01      0.641      0.064  
dst_day_ccy_flow_balance                 1.060000e-01      0.143      0.066  
src_day_ccy_log_matching_in_amount       1.300000e-02      0.360      0.004  
src_prior_ccy_amount_seen                9.410000e-01   

### 팀원 HTML 수치와 비교

| 팀원 수치 | 기간 | 이 노트북에서 |
|---|---|---|
| 송신 계좌에 처음 등장한 금액: 정상 10.72%, 무패턴 72.90% / ACH 19.79%, 100% | 9/26–10/4 표본 | `src_prior_ccy_amount_seen = 0` 비율 |
| 새로운 쌍·금액 조합: 정상 10.73%, 무패턴 72.90% | 앞 기간 표본 | `pair_amount_seen_before_day = 0` 비율 (9/26–10/4) |
| 송신 측 균형값 0: 정상 29.05%, 무패턴 66.90% | 앞 기간 | `src_day_ccy_flow_balance = 0` 비율 (9/26–10/4) |
| 송신 측 같은 금액 입금 > 0: 패턴 51.87%, 무패턴 0.59%, 정상 1.92% | 전수 | `src_day_ccy_log_matching_in_amount > 0` 비율 (분할 전체) |

팀원은 정상 **표본**(가중)으로 계산했으므로 소수점까지 같을 필요는 없다. 크게 다르면 정의가 다르다는 뜻이다.

In [11]:
P1 = "l.day BETWEEN DATE '2022-09-26' AND DATE '2022-10-04'"
cmp = con.execute(f"""
    SELECT l.cls, (l.fmt = 'ACH') AS ach, count(*) AS n,
           100 * avg((f.src_prior_ccy_amount_seen = 0)::INT)  AS src_new_amount_pct,
           100 * avg((f.pair_amount_seen_before_day = 0)::INT) AS pair_new_amount_pct,
           100 * avg((f.src_day_ccy_flow_balance = 0)::INT)    AS src_balance0_pct
    FROM feat f JOIN labels l USING (edge_id) WHERE {P1} AND l.cls IN (0, 2)
    GROUP BY GROUPING SETS ((l.cls), (l.cls, ach)) ORDER BY ach NULLS FIRST, l.cls
""").df()
display(cmp.round(2))
display(con.execute("""
    SELECT l.cls, 100 * avg((f.src_day_ccy_log_matching_in_amount > 0)::INT) AS src_matching_in_pct
    FROM feat f JOIN labels l USING (edge_id) GROUP BY 1 ORDER BY 1
""").df().round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cls,ach,n,src_new_amount_pct,pair_new_amount_pct,src_balance0_pct
0,0,<NA>,18603888,10.66,10.67,29.02
1,2,<NA>,8586,72.90,72.90,66.90
2,0,False,16477279,9.50,9.52,28.73
3,2,False,2521,7.70,7.70,79.41
4,0,True,2126609,19.58,19.58,31.27
5,2,True,6065,100.00,100.00,61.70


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cls,src_matching_in_pct
0,0,5.48
1,1,51.48
2,2,0.55


### 단변량 AUC — 팀원 10/9–13 수치와 비교

정상 vs 무패턴, 방향 보정(`max(AUC, 1−AUC)`), ACH / 비ACH. 정상은 5% 무작위 표본(`hash(edge_id)`)으로 계산한다.

In [12]:
TEAM_AUC = {  # (ACH, 비ACH) — HTML 의 10/9–13 방향보정 단변량 AUC
    "minute_sender_vs_prior_hour": (0.752, 0.884), "minute_receiver_vs_prior_hour": (0.642, 0.500),
    "src_day_ccy_flow_balance": (0.654, 0.803), "dst_day_ccy_flow_balance": (0.617, 0.502),
    "src_day_ccy_log_matching_in_amount": (0.502, 0.509),
    "src_prior_ccy_amount_seen": (0.894, 0.517), "dst_prior_ccy_amount_seen": (0.894, 0.517),
    "src_prior_ccy_amount_vs_mean": (0.562, 0.765), "dst_prior_ccy_amount_vs_mean": (0.663, 0.540),
    "src_day_ccy_rate_vs_prior": (0.855, 0.548), "dst_day_ccy_rate_vs_prior": (0.814, 0.523),
    "pair_amount_seen_before_day": (0.894, 0.517),
}


def auc(score, y):
    """동률 평균 순위 AUC"""
    r = pd.Series(score).rank(method="average").to_numpy()
    n1 = int(y.sum()); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else np.nan


s = con.execute(f"""
    SELECT l.cls, l.fmt = 'ACH' AS ach, {", ".join(f'f."{c}"' for c in FEATURES)}
    FROM feat f JOIN labels l USING (edge_id)
    WHERE l.day BETWEEN DATE '2022-10-09' AND DATE '2022-10-13'
      AND (l.cls = 2 OR (l.cls = 0 AND hash(l.edge_id) % 20 = 0))
""").df()
rows_auc = []
for c in FEATURES:
    r = {"feature": c}
    for ach, name in [(True, "ACH"), (False, "비ACH")]:
        d = s[s.ach == ach]
        a = auc(d[c].to_numpy(), (d.cls == 2).to_numpy().astype(int))
        r[name] = round(max(a, 1 - a), 3)
        if c in TEAM_AUC:
            r[f"팀원_{name}"] = TEAM_AUC[c][0 if ach else 1]
    rows_auc.append(r)
print(f"정상 표본 {int((s.cls == 0).sum()):,} · 무패턴 {int((s.cls == 2).sum()):,}")
pd.DataFrame(rows_auc).set_index("feature")

정상 표본 427,210 · 무패턴 4,937


,ACH,팀원_ACH,비ACH,팀원_비ACH
feature,,,,
minute_sender_vs_prior_hour,0.632,0.752,0.671,0.884
minute_receiver_vs_prior_hour,0.613,0.642,0.501,0.500
src_day_ccy_flow_balance,0.656,0.654,0.804,0.803
dst_day_ccy_flow_balance,0.622,0.617,0.501,0.502
src_day_ccy_log_matching_in_amount,0.503,0.502,0.509,0.509
src_prior_ccy_amount_seen,0.896,0.894,0.518,0.517
dst_prior_ccy_amount_seen,0.895,0.894,0.518,0.517
src_prior_ccy_amount_vs_mean,0.561,0.562,0.765,0.765
dst_prior_ccy_amount_vs_mean,0.662,0.663,0.540,0.540


In [13]:
# 쌍둥이 칸 확인: 같은 분 같은 쌍 다른 거래가 있는 행에서 is_max 와 라벨
con.execute("""
    SELECT f.split, l.cls, f.pair_same_minute_is_max AS is_max, count(*) AS n
    FROM feat f JOIN labels l USING (edge_id)
    WHERE f.pair_same_minute_other_n = 1 AND f.pair_count_7d = 0
    GROUP BY ALL ORDER BY ALL
""").df().pivot_table(index=["split", "is_max"], columns="cls", values="n", fill_value=0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cls                 0       1       2
split is_max                         
test  0.0     19915.0   373.0   739.0
      1.0     19207.0  1465.0   355.0
train 0.0     97911.0   665.0  1323.0
      1.0     94712.0  4594.0   593.0
val   0.0     13310.0   362.0   594.0
      1.0     12506.0  1469.0   291.0

In [14]:
meta = {"history": HISTORY, "splits": SPLITS, "rows": rows, "features": FEATURES,
        "team_features": TEAM_FEATURES, "pair_features": PAIR_FEATURES,
        "describe": desc.to_dict(orient="index"), "auc_1009_1013": rows_auc}
(OUT / "features.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
con.close()
print("완료:", OUT)
print("다음: 06 LightGBM — features_{split} (v10) + 05_features 를 edge_id 로 붙여 학습")

완료: /workspace/money_laundry_ai_/parquets/HI-Large/nonpaten_vs_normal/05_features
다음: 06 LightGBM — features_{split} (v10) + 05_features 를 edge_id 로 붙여 학습
